# Strides and Memory Layout

## Quick notes

### What this covers
How an ndarray is really stored (one flat byte buffer plus a header), what strides and contiguity mean, when you get a view versus a copy, how dtypes work at byte level (casting, overflow, float precision, nan/inf), and tools for raw memory: `as_strided`, `sliding_window_view`, `np.frombuffer`, `np.memmap`, `np.nditer`.

### Key points
- An array = data buffer + header (`dtype`, `shape`, `strides`, `flags`). Transpose, slice and reshape usually change only the header.
- `strides` are **bytes** to step along each axis. C order `(3,4)` int32 -> `(16, 4)`; `.T` just swaps them.
- C order (row-major) vs Fortran order (column-major) change memory layout, not the values you see. `order='F'` in `reshape`/`ravel` changes the reading order.
- `flags.c_contiguous` / `f_contiguous` / `owndata` / `writeable` describe the layout; `np.ascontiguousarray` makes a C-contiguous copy when needed.
- Views: basic slicing, `.T`, `.view()`, often `reshape`/`ravel`. Copies: fancy indexing, boolean masks, `.copy()`, `.flatten()`, `astype`.
- Check sharing with `.base`, `np.shares_memory` (exact) and `np.may_share_memory` (fast, bounds only).
- `sliding_window_view` gives safe read-only rolling windows; `as_strided` is the unsafe low-level version. Broadcasting is a stride-0 view.
- dtype: `kind`, `itemsize`, `byteorder`; `np.iinfo` / `np.finfo` give limits.
- `astype` converts values (copy); `view(dtype)` reinterprets the same bytes.
- `np.result_type`, `np.can_cast`, and `casting=` control and explain type promotion.
- Fixed-size integers wrap around silently; floats have limited precision (`eps`) and special values `nan`/`inf`.
- `np.errstate` controls float warnings (`ignore`, `warn`, `raise`).
- Memory cost = elements x itemsize; use smaller dtypes and in-place ops (`out=`, `+=`).
- `np.frombuffer` wraps bytes without copying; `np.memmap` maps a file as an array; `np.nditer` is the low-level iterator.

### Gotchas (the part worth revising twice)
- `b = a[idx_array]; b[0] = 1` does NOT change `a` (copy), but `a[idx_array] = 1` does. `a[[0,1]][0] = 5` writes to a throwaway copy.
- `reshape` / `ravel` return a view only when strides allow; otherwise they silently copy. Never depend on it for writing.
- `.view(np.int32)` on floats exposes raw bits; it does not convert. Use `astype` for values.
- `uint8` / `int8` arithmetic wraps with no error (`250 + 10 -> 4`). `sum(dtype=np.uint8)` wraps too. Widen first, then `np.clip`.
- Default integer size is platform-dependent; name the dtype when size matters.
- Python-scalar casting changed in NumPy 2.0 (NEP 50): on 1.26 `int8_array + 1000` upcasts to int16, on 2.x it overflows.
- `nan != nan`; comparisons with nan are `False`; `sum` with a nan gives nan (use `nansum`). Integer arrays cannot hold nan.
- Never compare floats with `==`; use `np.isclose` / `np.allclose`, and mind `atol` near zero.
- `as_strided` has no bounds checks: wrong arguments read garbage memory; windows overlap so writes hit several cells.
- `np.memmap` has no header: you must supply the correct dtype and shape. `np.zeros` of a huge shape may "work" lazily and crash later.

### When it matters
Image/audio pipelines (uint8 overflow), rolling-window features without Python loops, reading binary files bigger than RAM, debugging "why did my write not stick" bugs, and squeezing memory with float32/int8. Layout matters for speed on big arrays (iterate along the contiguous axis).

### One-line recap example
```python
import numpy as np
a = np.arange(6, dtype=np.int32).reshape(2, 3)
print(a.strides, a.T.strides, np.shares_memory(a, a.T))   # (12, 4) (4, 12) True
```

In [1]:
# 01. What an ndarray Really Is
# 02. Strides
# 03. C Order vs Fortran Order
# 04. Contiguity Flags
# 05. Views vs Copies
# 06. Checking Memory Sharing
# 07. Transpose, Reshape and Hidden Copies
# 08. as_strided and sliding_window_view
# 09. dtype Internals
# 10. view(dtype) vs astype
# 11. Casting Rules
# 12. Integer Overflow and Wraparound
# 13. Float Precision and eps
# 14. nan and inf
# 15. np.errstate
# 16. Memory Cost
# 17. np.frombuffer
# 18. memmap
# 19. nditer Basics
# 20. Cross-References
#
# Related: 01 (array basics), 03 (reshape), 04 (broadcasting - uses stride 0 trick),
# 08 (performance), 09 (file IO).

import os
import tempfile

import numpy as np
from numpy.lib.stride_tricks import as_strided, sliding_window_view

## 01. What an ndarray Really Is

- An ndarray = ONE flat block of bytes + a small header that says how to read it.
- The header holds: dtype, shape, strides, and flags.
- Same bytes can be read in many ways (transpose, slice, reshape) by only
  changing the header. That is why views are cheap.

In [2]:
a = np.arange(12, dtype=np.int32).reshape(3, 4)
print("a =\n", a)
print("shape =", a.shape, ", dtype =", a.dtype, ", itemsize =", a.itemsize, "bytes")
print("size =", a.size, ", nbytes =", a.nbytes, "(size * itemsize)")
print("ndim =", a.ndim)
print("# Gotcha: nbytes counts only the data buffer, not the small header object.")
print()

a =
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
shape = (3, 4) , dtype = int32 , itemsize = 4 bytes
size = 12 , nbytes = 48 (size * itemsize)
ndim = 2
# Gotcha: nbytes counts only the data buffer, not the small header object.



## 02. Strides

- strides = how many BYTES to jump to reach the next element along each axis.
- For a C-order int32 (3, 4): next column = 4 bytes, next row = 4 * 4 = 16 bytes.
- Address of a[i, j] = start + i * strides[0] + j * strides[1].

In [3]:
print("a.strides =", a.strides, "(row jump, column jump) in bytes")
print("a.T.strides =", a.T.strides, "(transpose just swaps the strides - no data moved)")
print("a[:, ::2].strides =", a[:, ::2].strides, "(every 2nd column -> column jump doubles)")
print("a[::2].strides =", a[::2].strides, "(every 2nd row -> row jump doubles)")
b = np.arange(5, dtype=np.int64)
print("b[::-1].strides =", b[::-1].strides, "(negative stride = reading backwards)")
print("# Gotcha: strides are in BYTES, not elements. Divide by itemsize for element steps.")
print()

a.strides = (16, 4) (row jump, column jump) in bytes
a.T.strides = (4, 16) (transpose just swaps the strides - no data moved)
a[:, ::2].strides = (16, 8) (every 2nd column -> column jump doubles)
a[::2].strides = (32, 4) (every 2nd row -> row jump doubles)
b[::-1].strides = (-8,) (negative stride = reading backwards)
# Gotcha: strides are in BYTES, not elements. Divide by itemsize for element steps.



## 03. C Order vs Fortran Order

- C order ('C', row-major): last axis changes fastest in memory.
- Fortran order ('F', column-major): first axis changes fastest.
- Same logical array, different memory layout. Indexing results are identical.

In [4]:
c = np.arange(6, dtype=np.int32).reshape(2, 3)
f = np.asfortranarray(c)
print("c (C order) =\n", c)
print("f (F order) =\n", f)
print("equal as arrays:", np.array_equal(c, f))
print("c.strides =", c.strides, ", f.strides =", f.strides)
print("memory of c:", c.ravel(order="K"), "(order='K' = as laid out in memory)")
print("memory of f:", f.ravel(order="K"))
print("f.ravel('C') =", f.ravel("C"), ", f.ravel('F') =", f.ravel("F"))
print("np.zeros((2,3), order='F').flags.f_contiguous =", np.zeros((2, 3), order="F").flags.f_contiguous)
print("reshape with order='F':", np.arange(6).reshape(2, 3, order="F").tolist())
print("# Gotcha: reshape/ravel read elements in C order by default, whatever the memory layout is.")
print("#   order='F' changes the READING order, not the memory.")
print()

c (C order) =
 [[0 1 2]
 [3 4 5]]
f (F order) =
 [[0 1 2]
 [3 4 5]]
equal as arrays: True
c.strides = (12, 4) , f.strides = (4, 8)
memory of c: [0 1 2 3 4 5] (order='K' = as laid out in memory)
memory of f: [0 3 1 4 2 5]
f.ravel('C') = [0 1 2 3 4 5] , f.ravel('F') = [0 3 1 4 2 5]
np.zeros((2,3), order='F').flags.f_contiguous = True
reshape with order='F': [[0, 2, 4], [1, 3, 5]]
# Gotcha: reshape/ravel read elements in C order by default, whatever the memory layout is.
#   order='F' changes the READING order, not the memory.



## 04. Contiguity Flags

- flags.c_contiguous / f_contiguous say whether the data is one unbroken block
  in that order. Slicing with a step or taking columns usually breaks it.
- Many fast C routines (and np.frombuffer / .view) need contiguous data.

In [5]:
print("a C/F contiguous:", a.flags.c_contiguous, a.flags.f_contiguous)
print("a.T C/F contiguous:", a.T.flags.c_contiguous, a.T.flags.f_contiguous)
print("a[:, :2] C/F contiguous:", a[:, :2].flags.c_contiguous, a[:, :2].flags.f_contiguous)
print("a[::2] C contiguous:", a[::2].flags.c_contiguous)
print("a[0] (one row) C contiguous:", a[0].flags.c_contiguous)
print("1-D b is both C and F contiguous:", b.flags.c_contiguous, b.flags.f_contiguous)
print("a.flags.owndata =", a.flags.owndata, ", a[:2].flags.owndata =", a[:2].flags.owndata)
print("a.flags.writeable =", a.flags.writeable)
d = np.ascontiguousarray(a.T)
print("np.ascontiguousarray(a.T) C contiguous:", d.flags.c_contiguous, "(made a real copy)")
print("# Gotcha: a.T of a C array is F-contiguous, not C-contiguous. Looping over it row by row")
print("#   jumps around memory and is slower on big arrays (cache misses).")
print()

a C/F contiguous: True False
a.T C/F contiguous: False True
a[:, :2] C/F contiguous: False False
a[::2] C contiguous: False
a[0] (one row) C contiguous: True
1-D b is both C and F contiguous: True True
a.flags.owndata = False , a[:2].flags.owndata = False
a.flags.writeable = True
np.ascontiguousarray(a.T) C contiguous: True (made a real copy)
# Gotcha: a.T of a C array is F-contiguous, not C-contiguous. Looping over it row by row
#   jumps around memory and is slower on big arrays (cache misses).



## 05. Views vs Copies

- VIEW: new header, SAME data buffer. Writing through a view changes the original.
- COPY: new data buffer. Independent.
- Basic slicing (a[1:3], a[:, 0], a[::2]), .T, .reshape (when possible), .view(), .ravel()
  (when possible) -> views.
- Fancy (integer-array) indexing, boolean masks, .copy(), .flatten(), astype -> copies.

In [6]:
x = np.arange(6)
v = x[1:4]
v[0] = 100
print("x after v[0]=100 (slice is a view):", x)
fancy = x[[1, 2, 3]]
fancy[0] = -1
print("x after fancy[0]=-1 (fancy index is a copy):", x)
mask_copy = x[x > 3]
mask_copy[:] = 0
print("x after writing to boolean-mask result:", x)
print("v.base is x:", v.base is x, ", x.base is None:", x.base is None)
print("x.flatten() is a copy, x.ravel() is a view when possible:",
      x.flatten().base is None, x.ravel().base is x)
# Writing INTO a fancy-indexed target does modify the original (it is an assignment).
x[[0, 1]] = 7
print("x after x[[0,1]] = 7 (assignment form works):", x)
print("# Gotcha: a[idx] = value modifies a, but b = a[idx]; b[...] = value does NOT.")
print("#   Chained fancy assignment like a[[0,1]][0] = 5 silently writes to a temporary copy.")
t = np.arange(4)
t[[0, 1]][0] = 55
print("t after t[[0,1]][0] = 55:", t, "(unchanged - the write went to a temporary)")
print()

x after v[0]=100 (slice is a view): [  0 100   2   3   4   5]
x after fancy[0]=-1 (fancy index is a copy): [  0 100   2   3   4   5]
x after writing to boolean-mask result: [  0 100   2   3   4   5]
v.base is x: True , x.base is None: True
x.flatten() is a copy, x.ravel() is a view when possible: True True
x after x[[0,1]] = 7 (assignment form works): [7 7 2 3 4 5]
# Gotcha: a[idx] = value modifies a, but b = a[idx]; b[...] = value does NOT.
#   Chained fancy assignment like a[[0,1]][0] = 5 silently writes to a temporary copy.
t after t[[0,1]][0] = 55: [0 1 2 3] (unchanged - the write went to a temporary)



## 06. Checking Memory Sharing

- arr.base: the array that owns the memory (None if arr owns its data).
- np.shares_memory(a, b): exact answer (can be slow for odd layouts).
- np.may_share_memory(a, b): fast bounds check only - may say True when the
  arrays interleave but never actually overlap.

In [7]:
base = np.arange(10)
ev = base[::2]
od = base[1::2]
print("np.shares_memory(base, ev):", np.shares_memory(base, ev))
print("np.shares_memory(ev, od):", np.shares_memory(ev, od), "(interleaved, no real overlap)")
print("np.may_share_memory(ev, od):", np.may_share_memory(ev, od), "(only compares memory bounds)")
print("np.shares_memory(base, base.copy()):", np.shares_memory(base, base.copy()))
print("ev.base is base:", ev.base is base)
vv = ev[1:]
print("view of a view: vv.base is base:", vv.base is base, "(base always points to the owner)")
print("# Gotcha: .base is None does NOT mean 'copy' for sure - np.arange(...).reshape(...) has a base")
print("#   but np.array(x) has none. Prefer np.shares_memory when you need a real answer.")
print()

np.shares_memory(base, ev): True
np.shares_memory(ev, od): False (interleaved, no real overlap)
np.may_share_memory(ev, od): True (only compares memory bounds)
np.shares_memory(base, base.copy()): False
ev.base is base: True
view of a view: vv.base is base: True (base always points to the owner)
# Gotcha: .base is None does NOT mean 'copy' for sure - np.arange(...).reshape(...) has a base
#   but np.array(x) has none. Prefer np.shares_memory when you need a real answer.



## 07. Transpose, Reshape and Hidden Copies

- .T and swapaxes are always views (strides swapped).
- reshape returns a view IF the strides allow it, otherwise silently copies.
- ravel() same rule; flatten() always copies.
- Setting arr.shape = ... instead of reshape raises an error if a copy would be needed.

In [8]:
m = np.arange(6).reshape(2, 3)
print("m.T is a view:", np.shares_memory(m, m.T))
print("m.reshape(3,2) is a view:", np.shares_memory(m, m.reshape(3, 2)))
print("m.T.reshape(6) is a view:", np.shares_memory(m, m.T.reshape(6)), "(needs a copy)")
print("m.T.ravel() =", m.T.ravel(), "(copy made silently)")
try:
    mt = m.T
    mt.shape = (6,)
except AttributeError as e:
    print("m.T.shape = (6,) ->", type(e).__name__, ":", str(e)[:60])
print("# Gotcha: whether reshape gives a view or copy depends on the layout, so never rely on")
print("#   'reshape then write' to modify the original. Check np.shares_memory if it matters.")
print()

m.T is a view: True
m.reshape(3,2) is a view: True
m.T.reshape(6) is a view: False (needs a copy)
m.T.ravel() = [0 3 1 4 2 5] (copy made silently)
m.T.shape = (6,) -> AttributeError : Incompatible shape for in-place modification. Use `.reshape(
# Gotcha: whether reshape gives a view or copy depends on the layout, so never rely on
#   'reshape then write' to modify the original. Check np.shares_memory if it matters.



<tmp> DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  mt.shape = (6,)


## 08. as_strided and sliding_window_view

- as_strided(x, shape, strides) builds ANY view you describe - no safety checks.
- Wrong shape/strides = reads memory outside the array = garbage or crash. Use with care.
- sliding_window_view(x, window) is the SAFE read-only version for rolling windows.
- Broadcasting itself is the stride-0 trick: np.broadcast_to makes a read-only view.

In [9]:
s = np.arange(6, dtype=np.int64)
manual = as_strided(s, shape=(4, 3), strides=(s.strides[0], s.strides[0]))
print("as_strided windows of length 3:\n", manual)
safe = sliding_window_view(s, 3)
print("sliding_window_view(s, 3):\n", safe)
print("same result:", np.array_equal(manual, safe))
print("rolling mean (window 3):", safe.mean(axis=1))
print("window view is read-only:", not safe.flags.writeable)
print("2-D windows shape:", sliding_window_view(np.arange(16).reshape(4, 4), (2, 2)).shape)
bt = np.broadcast_to(np.array([1, 2, 3]), (4, 3))
print("broadcast_to strides:", bt.strides, "(row stride 0 = same row reused), writeable:", bt.flags.writeable)
print("# Gotcha: windows overlap, so many 'rows' share the same bytes. Never write into an")
print("#   as_strided view; a single write changes several windows.")
print()

as_strided windows of length 3:
 [[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]]
sliding_window_view(s, 3):
 [[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]]
same result: True
rolling mean (window 3): [1. 2. 3. 4.]
window view is read-only: True
2-D windows shape: (3, 3, 2, 2)
broadcast_to strides: (0, 8) (row stride 0 = same row reused), writeable: False
# Gotcha: windows overlap, so many 'rows' share the same bytes. Never write into an
#   as_strided view; a single write changes several windows.



## 09. dtype Internals

- dtype describes how to read bytes: kind, size, byte order.
- kind codes: 'i' signed int, 'u' unsigned, 'f' float, 'c' complex, 'b' bool,
  'U' str, 'M' datetime64, 'm' timedelta64, 'O' object.
- byteorder: '<' little endian, '>' big endian, '=' native, '|' not applicable.
- itemsize = bytes per element.

In [10]:
for dt in (np.int8, np.uint16, np.int32, np.float32, np.float64, np.complex128, np.bool_):
    t_ = np.dtype(dt)
    print(f"{t_.name:<11} kind={t_.kind!r} itemsize={t_.itemsize} char={t_.char!r}")
little = np.dtype("<i4")
big = np.dtype(">i4")
print("'<i4' byteorder:", little.byteorder, ", '>i4' byteorder:", big.byteorder)
arr_big = np.array([1, 256], dtype=">i4")
print("big-endian bytes of [1, 256]:", arr_big.tobytes().hex())
arr_little = np.array([1, 256], dtype="<i4")
print("little-endian bytes of [1, 256]:", arr_little.tobytes().hex())
print("values equal despite different bytes:", np.array_equal(arr_big, arr_little))
print("np.dtype('U5').itemsize =", np.dtype("U5").itemsize, "(4 bytes per char)")
print("np.dtype('S5').itemsize =", np.dtype("S5").itemsize, "(1 byte per char)")
print("np.dtype('O').itemsize is a pointer size - object arrays hold references, not values.")
print("np.iinfo(np.int16):", np.iinfo(np.int16).min, "to", np.iinfo(np.int16).max)
print("np.iinfo(np.uint8):", np.iinfo(np.uint8).min, "to", np.iinfo(np.uint8).max)
print("# Gotcha: np.int_ / default integer size depends on platform (64-bit on most Linux/macOS,")
print("#   32-bit on older Windows builds). Always name the dtype when size matters.")
print()

int8        kind='i' itemsize=1 char='b'
uint16      kind='u' itemsize=2 char='H'
int32       kind='i' itemsize=4 char='i'
float32     kind='f' itemsize=4 char='f'
float64     kind='f' itemsize=8 char='d'
complex128  kind='c' itemsize=16 char='D'
bool        kind='b' itemsize=1 char='?'
'<i4' byteorder: = , '>i4' byteorder: >
big-endian bytes of [1, 256]: 0000000100000100
little-endian bytes of [1, 256]: 0100000000010000
values equal despite different bytes: True
np.dtype('U5').itemsize = 20 (4 bytes per char)
np.dtype('S5').itemsize = 5 (1 byte per char)
np.dtype('O').itemsize is a pointer size - object arrays hold references, not values.
np.iinfo(np.int16): -32768 to 32767
np.iinfo(np.uint8): 0 to 255
# Gotcha: np.int_ / default integer size depends on platform (64-bit on most Linux/macOS,
#   32-bit on older Windows builds). Always name the dtype when size matters.



## 10. view(dtype) vs astype

- astype(new): CONVERTS the values (makes a copy).
- view(new): REINTERPRETS the same bytes as another dtype (no copy, shape may change).
- view needs compatible sizes and a contiguous last axis.

In [11]:
fl = np.array([1.0, 2.0], dtype=np.float32)
print("astype(int32):", fl.astype(np.int32), "(values converted)")
print("view(int32):", fl.view(np.int32), "(raw bit patterns of the floats)")
print("float32 1.0 in hex bits:", hex(int(fl.view(np.uint32)[0])))
u8 = np.array([1, 0, 0, 0, 2, 0, 0, 0], dtype=np.uint8)
print("uint8 x8 viewed as int32:", u8.view("<i4"), "(shape changed: 8 -> 2)")
print("view shares memory:", np.shares_memory(u8, u8.view("<i4")))
print("astype shares memory:", np.shares_memory(fl, fl.astype(np.int32)))
print("astype(copy=False) same dtype reuses the array:", fl.astype(np.float32, copy=False) is fl)
print("# Gotcha: .view(np.int32) on float data does NOT round numbers. It exposes raw bits.")
print("#   Use astype for value conversion, view only when you mean 'same bytes, new meaning'.")
print()

astype(int32): [1 2] (values converted)
view(int32): [1065353216 1073741824] (raw bit patterns of the floats)
float32 1.0 in hex bits: 0x3f800000
uint8 x8 viewed as int32: [1 2] (shape changed: 8 -> 2)
view shares memory: True
astype shares memory: False
astype(copy=False) same dtype reuses the array: True
# Gotcha: .view(np.int32) on float data does NOT round numbers. It exposes raw bits.
#   Use astype for value conversion, view only when you mean 'same bytes, new meaning'.



## 11. Casting Rules

- np.result_type tells what dtype an operation will produce.
- Mixing int and float -> float. Mixing bool and int -> int.
- Python scalars are "weak" in NumPy 1.x value-based casting: int8_array + 1000 may upcast.
- astype(..., casting='safe') refuses lossy conversions; default for astype is 'unsafe'.
- np.can_cast checks without doing it.

In [12]:
print("result_type(int8, int16):", np.result_type(np.int8, np.int16))
print("result_type(int32, float32):", np.result_type(np.int32, np.float32), "(float32 can't hold all int32)")
print("result_type(int64, float32):", np.result_type(np.int64, np.float32))
print("result_type(bool, int8):", np.result_type(np.bool_, np.int8))
print("result_type(uint8, int8):", np.result_type(np.uint8, np.int8), "(signed + unsigned -> bigger signed)")
print("can_cast(int32 -> float64, 'safe'):", np.can_cast(np.int32, np.float64, "safe"))
print("can_cast(float64 -> int32, 'safe'):", np.can_cast(np.float64, np.int32, "safe"))
print("can_cast(float64 -> int32, 'same_kind'):", np.can_cast(np.float64, np.int32, "same_kind"))
try:
    np.array([1.5, 2.5]).astype(np.int32, casting="safe")
except TypeError as e:
    print("astype(casting='safe') ->", type(e).__name__)
print("float -> int truncates toward zero:", np.array([1.9, -1.9]).astype(np.int32))
print("int8 array + 1 stays:", (np.array([1], dtype=np.int8) + 1).dtype)
try:
    print("int8 array + 1000 (value-based, NumPy 1.x):", (np.array([1], dtype=np.int8) + 1000).dtype)
except OverflowError:
    print("NumPy 2 rejects a Python scalar outside the array dtype range")
print("true division of ints gives:", (np.array([4, 2], dtype=np.int32) / 2).dtype)
print("# Gotcha: casting rules for Python scalars changed in NumPy 2.0 (NEP 50). On 1.26 the")
print("#   value decides the type; on 2.x the array dtype wins and 1000 would overflow int8.")
print()

result_type(int8, int16): int16
result_type(int32, float32): float64 (float32 can't hold all int32)
result_type(int64, float32): float64
result_type(bool, int8): int8
result_type(uint8, int8): int16 (signed + unsigned -> bigger signed)
can_cast(int32 -> float64, 'safe'): True
can_cast(float64 -> int32, 'safe'): False
can_cast(float64 -> int32, 'same_kind'): False
astype(casting='safe') -> TypeError
float -> int truncates toward zero: [ 1 -1]
int8 array + 1 stays: int8
NumPy 2 rejects a Python scalar outside the array dtype range
true division of ints gives: float64
# Gotcha: casting rules for Python scalars changed in NumPy 2.0 (NEP 50). On 1.26 the
#   value decides the type; on 2.x the array dtype wins and 1000 would overflow int8.



## 12. Integer Overflow and Wraparound

- Fixed-size integers WRAP AROUND silently with no error (array math).
- Python ints never overflow, so lists and arrays behave differently.
- Use a bigger dtype (or float) for sums, products and accumulators.

In [13]:
u = np.array([250, 255], dtype=np.uint8)
print("uint8 [250, 255] + 10 =", u + np.uint8(10), "(wraps past 255)")
i = np.array([127], dtype=np.int8)
print("int8 127 + 1 =", i + np.int8(1), "(wraps to -128)")
print("uint8 0 - 1 =", np.array([0], dtype=np.uint8) - np.uint8(1))
print("int8 abs(-128) =", np.abs(np.array([-128], dtype=np.int8)), "(no +128 in int8)")
img = np.array([200, 100], dtype=np.uint8)
print("uint8 sum of [200, 100] with default accumulator:", img.sum(), ", dtype:", img.sum().dtype.kind)
print("uint8 sum with dtype=np.uint8:", img.sum(dtype=np.uint8), "(wrapped: 300 - 256)")
print("uint8 mean (uses float64 internally):", img.mean())
print("uint8 [200] * 2 =", np.array([200], dtype=np.uint8) * np.uint8(2))
big_prod = np.array([3_000_000_000], dtype=np.int64) * 4
print("int64 3e9 * 4 =", big_prod, "(fine in int64)")
print("int32 same:", np.array([3_000_000_000 // 3], dtype=np.int32) * np.int32(4), "(wrapped - 1e9*4 > 2^31)")
print("python int 2**64 + 1 =", 2**64 + 1)
try:
    np.array([2**64])
    print("np.array([2**64]).dtype =", np.array([2**64]).dtype, "(falls back to object)")
except OverflowError as e:
    print("OverflowError:", e)
print("# Gotcha: image arithmetic in uint8 (brightness + 50) wraps instead of clipping. Cast to")
print("#   int16/float first, then np.clip(..., 0, 255).astype(np.uint8).")
print()

uint8 [250, 255] + 10 = [4 9] (wraps past 255)
int8 127 + 1 = [-128] (wraps to -128)
uint8 0 - 1 = [255]
int8 abs(-128) = [-128] (no +128 in int8)
uint8 sum of [200, 100] with default accumulator: 300 , dtype: u
uint8 sum with dtype=np.uint8: 44 (wrapped: 300 - 256)
uint8 mean (uses float64 internally): 150.0
uint8 [200] * 2 = [144]
int64 3e9 * 4 = [12000000000] (fine in int64)
int32 same: [-294967296] (wrapped - 1e9*4 > 2^31)
python int 2**64 + 1 = 18446744073709551617
np.array([2**64]).dtype = object (falls back to object)
# Gotcha: image arithmetic in uint8 (brightness + 50) wraps instead of clipping. Cast to
#   int16/float first, then np.clip(..., 0, 255).astype(np.uint8).



## 13. Float Precision and eps

- float64 has ~15-16 decimal digits, float32 ~7. np.finfo tells the limits.
- eps = gap between 1.0 and the next float. Not a general 'error size' - gaps grow with magnitude.
- Never test floats with ==. Use np.isclose / np.allclose.

In [14]:
print("0.1 + 0.2 == 0.3:", 0.1 + 0.2 == 0.3)
print("np.isclose(0.1 + 0.2, 0.3):", np.isclose(0.1 + 0.2, 0.3))
print("float64 eps:", np.finfo(np.float64).eps)
print("float32 eps:", np.finfo(np.float32).eps)
print("float64 max:", np.finfo(np.float64).max)
print("float64 smallest normal:", np.finfo(np.float64).tiny)
print("float32 1e8 + 1 == 1e8:", np.float32(1e8) + np.float32(1) == np.float32(1e8), "(1 is below the gap)")
print("float64 1e16 + 1 == 1e16:", 1e16 + 1 == 1e16)
print("np.spacing(1.0):", np.spacing(1.0), ", np.spacing(1e10):", np.spacing(1e10))
print("np.nextafter(1.0, 2.0) - 1.0:", np.nextafter(1.0, 2.0) - 1.0)
print("float32(0.1) printed as float64:", float(np.float32(0.1)))
tiny = np.full(10**5, 0.1, dtype=np.float32)
print("sum of 1e5 float32 0.1 (pairwise summation):", float(tiny.sum()), "vs exact 10000.0")
print("allclose with rtol/atol:", np.allclose([1.0, 1e-9], [1.0 + 1e-9, 0.0], rtol=1e-5, atol=1e-8))
print("np.round(2.5), np.round(3.5), np.round(-0.5):", np.round(2.5), np.round(3.5), np.round(-0.5), "(banker's rounding)")
print("# Gotcha: isclose is not symmetric and atol matters near 0. isclose(1e-10, 0) is True by")
print("#   default atol=1e-8, which may be too loose for tiny-scale data.")
print()

0.1 + 0.2 == 0.3: False
np.isclose(0.1 + 0.2, 0.3): True
float64 eps: 2.220446049250313e-16
float32 eps: 1.1920929e-07
float64 max: 1.7976931348623157e+308
float64 smallest normal: 2.2250738585072014e-308
float32 1e8 + 1 == 1e8: True (1 is below the gap)
float64 1e16 + 1 == 1e16: True
np.spacing(1.0): 2.220446049250313e-16 , np.spacing(1e10): 1.9073486328125e-06
np.nextafter(1.0, 2.0) - 1.0: 2.220446049250313e-16
float32(0.1) printed as float64: 0.10000000149011612
sum of 1e5 float32 0.1 (pairwise summation): 10000.0009765625 vs exact 10000.0
allclose with rtol/atol: True
np.round(2.5), np.round(3.5), np.round(-0.5): 2.0 4.0 -0.0 (banker's rounding)
# Gotcha: isclose is not symmetric and atol matters near 0. isclose(1e-10, 0) is True by
#   default atol=1e-8, which may be too loose for tiny-scale data.



## 14. nan and inf

- nan = 'not a number' (undefined result). inf = overflow or division by zero.
- nan != nan, so use np.isnan. Any arithmetic with nan gives nan.
- nan is a float; integer arrays cannot hold it.

In [15]:
n = np.array([1.0, np.nan, 3.0, np.inf])
print("n =", n)
print("nan == nan:", np.nan == np.nan)
print("np.isnan(n):", np.isnan(n))
print("np.isinf(n):", np.isinf(n))
print("np.isfinite(n):", np.isfinite(n))
print("n.sum():", n.sum(), ", np.nansum(n):", np.nansum(n))
print("np.nanmax(n[np.isfinite(n)]):", np.nanmax(n[np.isfinite(n)]))
print("max with nan:", np.max(np.array([1.0, np.nan])), "| np.fmax:", np.fmax(1.0, np.nan))
print("inf - inf:", np.inf - np.inf, ", inf * 0:", np.inf * 0, ", 1/inf:", 1 / np.inf)
print("nan in a comparison:", np.nan < 1, ", np.nan > 1:", np.nan > 1)
print("np.nan_to_num(n):", np.nan_to_num(n, nan=0.0, posinf=999.0))
print("-0.0 == 0.0:", -0.0 == 0.0, ", np.signbit(-0.0):", np.signbit(-0.0))
print("np.unique treats nans as equal (1.21+):", np.unique(np.array([np.nan, np.nan, 1.0])))
try:
    np.array([1, 2, 3], dtype=np.int32)[0] = np.nan
except ValueError as e:
    print("int array = nan ->", type(e).__name__)
print("# Gotcha: nan in a boolean mask silently makes the comparison False: arr > 0 drops nan rows.")
print("#   Use ~np.isnan(arr) deliberately when cleaning data.")
print()

n = [ 1. nan  3. inf]
nan == nan: False
np.isnan(n): [False  True False False]
np.isinf(n): [False False False  True]
np.isfinite(n): [ True False  True False]
n.sum(): nan , np.nansum(n): inf
np.nanmax(n[np.isfinite(n)]): 3.0
max with nan: nan | np.fmax: 1.0
inf - inf: nan , inf * 0: nan , 1/inf: 0.0
nan in a comparison: False , np.nan > 1: False
np.nan_to_num(n): [  1.   0.   3. 999.]
-0.0 == 0.0: True , np.signbit(-0.0): True
np.unique treats nans as equal (1.21+): [ 1. nan]
int array = nan -> ValueError
# Gotcha: nan in a boolean mask silently makes the comparison False: arr > 0 drops nan rows.
#   Use ~np.isnan(arr) deliberately when cleaning data.



## 15. np.errstate

- NumPy float errors (divide by zero, overflow, underflow, invalid) give a warning
  and keep going. np.errstate sets what to do inside a with-block.
- Choices: 'ignore', 'warn', 'raise', 'call', 'print', 'log'.
- Integer division by zero is a separate case (also warns, returns 0).

In [16]:
x0 = np.array([1.0, 0.0])
with np.errstate(divide="ignore", invalid="ignore"):
    print("1/[1,0] ignoring errors:", 1.0 / x0)
    print("0/0:", np.float64(0.0) / np.float64(0.0))
try:
    with np.errstate(divide="raise"):
        1.0 / x0
except FloatingPointError as e:
    print("divide='raise' ->", type(e).__name__, ":", e)
try:
    with np.errstate(over="raise"):
        np.float32(3e38) * np.float32(10)
except FloatingPointError as e:
    print("over='raise' ->", e)
with np.errstate(all="ignore"):
    print("int divide by zero ignored:", np.array([5], dtype=np.int32) // np.array([0], dtype=np.int32))
print("np.geterr() default keys:", sorted(np.geterr()))
print("# Gotcha: errstate is per-thread and only for floating point. Integer overflow never")
print("#   triggers it (see section 12). Use np.seterr(all='raise') in tests to catch bugs early.")
print()

1/[1,0] ignoring errors: [ 1. inf]
0/0: nan
divide='raise' -> FloatingPointError : divide by zero encountered in divide
over='raise' -> overflow encountered in scalar multiply
int divide by zero ignored: [0]
np.geterr() default keys: ['divide', 'invalid', 'over', 'under']
# Gotcha: errstate is per-thread and only for floating point. Integer overflow never
#   triggers it (see section 12). Use np.seterr(all='raise') in tests to catch bugs early.



## 16. Memory Cost

- Memory = number of elements * itemsize. Pick the smallest dtype that fits the data.
- Intermediate temporaries cost too: a*b+c builds a*b first.
- In-place ops (out=, +=) avoid temporaries.

In [17]:
n_items = 1_000_000
for dt in (np.float64, np.float32, np.float16, np.int8, np.bool_):
    print(f"{n_items} x {np.dtype(dt).name:<8}: {np.dtype(dt).itemsize * n_items / 1e6:.1f} MB")
print("object arrays store one pointer per slot PLUS the Python objects themselves - far heavier than int64.")
print("a.T.nbytes =", a.T.nbytes, "(logical size - a view allocates no new data, yet nbytes still reports it)")
w = np.ones(5)
w2 = w * 2
before = w
w *= 2
print("w *= 2 keeps the same array object:", before is w, "| w * 2 builds a new one:", w2 is not w)
out = np.empty(5)
np.add(w, w, out=out)
print("np.add(..., out=out) =", out)
print("float16 range max:", np.finfo(np.float16).max, ", eps:", np.finfo(np.float16).eps)
print("# Gotcha: a huge np.zeros((100000, 100000)) may succeed instantly (lazy pages) and then")
print("#   crash when you write to it. Estimate nbytes BEFORE allocating.")
print()

1000000 x float64 : 8.0 MB
1000000 x float32 : 4.0 MB
1000000 x float16 : 2.0 MB
1000000 x int8    : 1.0 MB
1000000 x bool    : 1.0 MB
object arrays store one pointer per slot PLUS the Python objects themselves - far heavier than int64.
a.T.nbytes = 48 (logical size - a view allocates no new data, yet nbytes still reports it)
w *= 2 keeps the same array object: True | w * 2 builds a new one: True
np.add(..., out=out) = [4. 4. 4. 4. 4.]
float16 range max: 6.55e+04 , eps: 0.000977
# Gotcha: a huge np.zeros((100000, 100000)) may succeed instantly (lazy pages) and then
#   crash when you write to it. Estimate nbytes BEFORE allocating.



## 17. np.frombuffer

- np.frombuffer(buf, dtype) wraps existing bytes WITHOUT copying.
- From bytes -> read-only. From bytearray -> writable and linked to the buffer.
- Compare np.fromstring (deprecated for binary) - use frombuffer.

In [18]:
raw = (1).to_bytes(4, "little") + (2).to_bytes(4, "little") + (3).to_bytes(4, "little")
fb = np.frombuffer(raw, dtype="<i4")
print("frombuffer(bytes):", fb, ", writeable:", fb.flags.writeable)
ba = bytearray(raw)
fw = np.frombuffer(ba, dtype="<i4")
fw[0] = 99
print("frombuffer(bytearray) after write:", fw, "| bytearray first 4 bytes:", list(ba[:4]))
print("count/offset: ", np.frombuffer(raw, dtype="<i4", count=2, offset=4))
print("tobytes round trip:", np.array_equal(np.frombuffer(np.arange(4, dtype="<f8").tobytes(), dtype="<f8"), np.arange(4)))
print("memoryview of array has shape:", memoryview(np.arange(6, dtype=np.int32).reshape(2, 3)).shape)
try:
    np.frombuffer(b"\x00\x01\x02", dtype="<i4")
except ValueError as e:
    print("bad length ->", type(e).__name__, ":", str(e))
print("# Gotcha: the result keeps the source buffer alive and linked. Changing the bytearray")
print("#   changes the array. Use .copy() if you want independence.")
print()

frombuffer(bytes): [1 2 3] , writeable: False
frombuffer(bytearray) after write: [99  2  3] | bytearray first 4 bytes: [99, 0, 0, 0]
count/offset:  [2 3]
tobytes round trip: True
memoryview of array has shape: (2, 3)
bad length -> ValueError : buffer size must be a multiple of element size
# Gotcha: the result keeps the source buffer alive and linked. Changing the bytearray
#   changes the array. Use .copy() if you want independence.



## 18. memmap

- np.memmap maps a FILE as an array; only pages you touch are loaded.
- Good for arrays bigger than RAM. Modes: 'r' read, 'r+' read/write, 'w+' create, 'c' copy-on-write.
- Always flush() or delete the object so data reaches disk.
- Compare with 09 (file IO): np.save/np.load reads everything into memory.

In [19]:
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "data.dat")
    mm = np.memmap(path, dtype="<f4", mode="w+", shape=(4, 3))
    mm[:] = np.arange(12, dtype=np.float32).reshape(4, 3)
    mm[1, 1] = -1.0
    mm.flush()
    print("memmap shape/dtype:", mm.shape, mm.dtype, ", file size bytes:", os.path.getsize(path), "(4*3*4 = 48)")
    del mm
    ro = np.memmap(path, dtype="<f4", mode="r", shape=(4, 3))
    print("read back:\n", ro)
    print("is ndarray subclass:", isinstance(ro, np.ndarray), ", writeable:", ro.flags.writeable)
    part = np.memmap(path, dtype="<f4", mode="r", offset=12, shape=(2,))
    print("offset=12 bytes skips 3 floats:", part)
    cow = np.memmap(path, dtype="<f4", mode="c", shape=(4, 3))
    cow[0, 0] = 777.0
    print("copy-on-write changed memory only:", cow[0, 0], "| file still has:", ro[0, 0])
    print("np.fromfile alternative:", np.fromfile(path, dtype="<f4")[:4])
    del ro, part, cow
print("temp directory removed after the with-block.")
print("# Gotcha: you MUST give dtype and shape yourself - the file has no header. A wrong dtype")
print("#   gives silent garbage. On Windows an open memmap blocks deleting the file.")
print()

memmap shape/dtype: (4, 3) float32 , file size bytes: 48 (4*3*4 = 48)
read back:
 [[ 0.  1.  2.]
 [ 3. -1.  5.]
 [ 6.  7.  8.]
 [ 9. 10. 11.]]
is ndarray subclass: True , writeable: False
offset=12 bytes skips 3 floats: [ 3. -1.]
copy-on-write changed memory only: 777.0 | file still has: 0.0
np.fromfile alternative: [0. 1. 2. 3.]
temp directory removed after the with-block.
# Gotcha: you MUST give dtype and shape yourself - the file has no header. A wrong dtype
#   gives silent garbage. On Windows an open memmap blocks deleting the file.



## 19. nditer Basics

- np.nditer walks array elements efficiently, handling broadcasting and memory order.
- Default iteration order is 'K' = memory order (fastest), not necessarily C order.
- Use flags=['multi_index'] to get indexes, op_flags=['readwrite'] to modify.
- For most jobs ufuncs or np.ndenumerate are simpler; nditer is the low-level tool.

In [20]:
g = np.arange(6).reshape(2, 3)
print("C-order iteration:", [int(v_) for v_ in np.nditer(g, order="C")])
print("F-order iteration:", [int(v_) for v_ in np.nditer(g, order="F")])
print("default on g.T (memory order):", [int(v_) for v_ in np.nditer(g.T)], "vs g.T.ravel():", g.T.ravel().tolist())
it = np.nditer(g, flags=["multi_index"])
print("with multi_index:", [(it.multi_index, int(v_)) for v_ in it])
h = np.arange(4)
with np.nditer(h, op_flags=["readwrite"]) as it2:
    for cell in it2:
        cell[...] = cell * 10
print("modified in place:", h)
print("two operands broadcast together:", [(int(p), int(q)) for p, q in np.nditer([np.arange(3), np.array([[10], [20]])])])
print("ndenumerate:", [(idx, int(v_)) for idx, v_ in np.ndenumerate(np.array([[5, 6], [7, 8]]))])
print("# Gotcha: nditer's default order can differ from what you expect for transposed arrays.")
print("#   Pass order='C' when the order matters. In a Python loop it is still slow - prefer ufuncs.")
print()

C-order iteration: [0, 1, 2, 3, 4, 5]
F-order iteration: [0, 3, 1, 4, 2, 5]
default on g.T (memory order): [0, 1, 2, 3, 4, 5] vs g.T.ravel(): [0, 3, 1, 4, 2, 5]
with multi_index: [((0, 0), 0), ((0, 1), 1), ((0, 2), 2), ((1, 0), 3), ((1, 1), 4), ((1, 2), 5)]
modified in place: [ 0 10 20 30]
two operands broadcast together: [(0, 10), (1, 10), (2, 10), (0, 20), (1, 20), (2, 20)]
ndenumerate: [((0, 0), 5), ((0, 1), 6), ((1, 0), 7), ((1, 1), 8)]
# Gotcha: nditer's default order can differ from what you expect for transposed arrays.
#   Pass order='C' when the order matters. In a Python loop it is still slow - prefer ufuncs.



## 20. Cross-References

- 03-Reshaping-And-Manipulation: reshape/ravel/transpose (here: when they copy).
- 04-Operations-And-Broadcasting: broadcasting (here: stride 0 behind it).
- 08-Performance-And-Vectorization: speed (here: why layout and dtype affect it).
- 09-File-IO: save/load (here: memmap and frombuffer for raw files).
- 12-Advanced-Indexing-Ufuncs-And-Sorting: take/put and ufunc methods with out=.

In [21]:
print("Quick checklist: shape, dtype, strides, flags.owndata, np.shares_memory - five questions that explain most surprises.")

Quick checklist: shape, dtype, strides, flags.owndata, np.shares_memory - five questions that explain most surprises.
